#Notebook da Camada Silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]


In [0]:
def normalizar_texto(col):
    r"""
    Normaliza texto:
    - tudo minúsculo
    - mantém só letras (com acento) e números
    - todo o resto (pontuação, símbolos, '_', \n, \t, \r e outros espaços) vira espaço
    - sequências de espaço viram um só, e trim nas pontas
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    return F.trim(
        F.regexp_replace(F.lower(c.cast("string")), r"[^\p{L}\p{M}0-9]+", " ")
    )

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.chamados")

dftest = df_bronze_chamados.withColumn("canal", normalizar_texto("canal"))
display(dftest.groupBy("canal").count().orderBy(F.desc("count")))